In [45]:
%use kandy

We are trying to estimate the performance ratio of stdlib functions in Kotlin/Native vs. JVM. For this example, we focus
on `String.substring` method used in this particular scenario. We need to exclude the effect of GC to get the performace
ratio of execution time of the function itself. The following cell describes, how we collect the data. We
end up with a list of execution times.

In [46]:
import java.nio.file.Files
import java.nio.file.Path
import java.nio.file.StandardOpenOption

val kotlincNative = System.getProperty("user.home") + "/.konan/kotlin-native-prebuilt-linux-x86_64-2.2.10/bin/kotlinc-native"
val tempDir = Files.createTempDirectory("knative-perf")

fun runProcess(vararg command: String): String =
    ProcessBuilder(command.toList())
        .directory(tempDir.toFile())
        .start()
        .let { process ->
            process.inputStream.bufferedReader().readText()
                .also {
                    if (process.waitFor() != 0) {
                        process.errorStream.bufferedReader().readText().let(::println)
                        throw Exception("Process failed with code ${process.exitValue()}")
                    }
                }
        }

val source = """
import kotlin.concurrent.Volatile
import kotlin.time.measureTime
import kotlin.time.DurationUnit
import kotlin.time.Duration

@Volatile
private var blackhole: Any? = null
@Volatile
private var string = "abc".repeat(100000)

fun main() {
    repeat(1_000_000) { blackhole = string.substring(1000, 5000) }
    val iterations = 100_000
    val times = ArrayList<Duration>(iterations)
    repeat(iterations) {
        measureTime {
            repeat(100) {
                blackhole = string.substring(1000, 5000)
            }
        }.let { times.add(it) }
    }
    times.forEach { println(it.toDouble(DurationUnit.SECONDS)) }
}
"""

val sourceFile = tempDir.resolve("benchmark.kt")
val outputBinary = tempDir.resolve("benchmark.kexe")

Files.writeString(
    sourceFile,
    source,
    StandardOpenOption.CREATE,
)

runProcess(
        kotlincNative,
        sourceFile.toString(),
        "-o", outputBinary.toString()
)

val times = runProcess(outputBinary.toString())
    .split("\n")
    .mapNotNull { it.takeIf { it.isNotEmpty() }?.toDouble() }
    .toList()

For the exclusion of GC, we assume that the execution time distribution is bimodal (i .e. there are two peaks). The
first peak is formed by the executions that didn't trigger GC. The second peak is formed by the executions that
triggered GC. By finding a threshold for the execution time that lies between the two peaks, we can separate the
executions into those that ran GC and those which didn't.

In [47]:
val threshold = 0.00005

In [48]:
plot {
    histogram(times, binsOption = BinsOption.byNumber(100))
    vLine {
        xIntercept.constant(threshold)
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="9RbAz6"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("9RbAz6");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"x",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count"
},
"stat":"identity",
"data":{
"count":[61054.0,3467.0,5987.0,2426.0,1838.0,1672.0,1727.0,1607.0,1393.0,1632.0,851.0,1794.0,1504.0,1290.0,1715.0,1488.0,1502.0,950.0,1071.0,1140.0,903.0,616.0,286.0,211.0,222.0,206.0,156.0,110.0,96.0,105.0,68.0,94.0,78.0,85.0,66.0,72.0,60.0,50.0,35.0,52.0,49.0,59.0,43.0,24.0,22.0,15.0,13.0,5.0,3.0,10.0,9.0,8.0,4.0,0.0,4.0,0.0,1.0,3.0,1.0,3.0,4.0,2.0,2.0,4.0,4.0,1.0,2.0,2.0,1.0,4.0,3.0,1.0,2.0,2.0,1.0,2.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0],
"x":[1.3802599999999999E-5,4.1407799999999993E-5,6.901299999999998E-5,9.661819999999999E-5,1.242234E-4,1.518286E-4,1.7943379999999998E-4,2.0703899999999998E-4,2.3464419999999999E-4,2.622494E-4,2.8985459999999997E-4,3.1745979999999997E-4,3.45065E-4,3.7267019999999993E-4,4.002754E-4,4.2788059999999994E-4,4.554858E-4,4.8309099999999995E-4,5.106962E-4,5.383014E-4,5.659065999999999E-4,5.935118E-4,6.21117E-4,6.487221999999999E-4,6.763273999999999E-4,7.039326E-4,7.315377999999999E-4,7.591429999999999E-4,7.867481999999999E-4,8.143534E-4,8.419586E-4,8.695637999999999E-4,8.97169E-4,9.247741999999999E-4,9.523793999999999E-4,9.799846E-4,0.0010075898,0.001035195,0.0010628001999999999,0.0010904054,0.0011180105999999998,0.0011456157999999998,0.001173221,0.0012008262,0.0012284314,0.0012560365999999999,0.0012836418,0.0013112469999999998,0.0013388521999999998,0.0013664573999999999,0.0013940626,0.0014216678,0.0014492729999999998,0.0014768782,0.0015044834,0.0015320885999999998,0.0015596937999999999,0.001587299,0.0016149042,0.0016425093999999998,0.0016701145999999999,0.0016977198,0.0017253249999999998,0.0017529301999999998,0.0017805354,0.0018081406,0.0018357457999999998,0.0018633509999999999,0.0018909562,0.0019185613999999998,0.0019461665999999998,0.0019737718,0.002001377,0.0020289822,0.0020565873999999996,0.0020841925999999997,0.0021117977999999997,0.002139403,0.0021670082,0.0021946134,0.0022222186,0.0022498238,0.0022774289999999997,0.0023050341999999997,0.0023326393999999998,0.0023602446,0.0023878498,0.002415455,0.0024430601999999996,0.0024706653999999996,0.0024982705999999997,0.0025258757999999997,0.002553481,0.0025810862,0.0026086914,0.0026362966,0.0026639017999999996,0.0026915069999999997,0.0027191121999999997,0.0027467174]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"int",
"column":"count"
}]
}
},{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"xintercept":5.0E-5,
"geom":"vline",
"data":{
}
}],
"spec_id":"14"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED'

We then estimate the total time spent in GC by the following formula.

In [49]:
val (noGC, GC) = times.partition { it < threshold }
val avgNoGC = noGC.average()
GC.sumOf { it - avgNoGC } / times.sum()

0.8434806251864089

For measurements that run at least 100 000 iterations and don't print the times throught the execution (but all at once at the end), we estimate that GC is responsible for over 80% of the execution. That is insane. Therefore, we question the results.

* Can GC in K/N be responsible for such a high percentage of execution time? After all, this is very memory heavy benchmark.
* Is there an error in the benchmark setup, collection of the data or the processing?
* Is any one of our assumptions incorrect?
* Is this method for estimating GC time sound?